In [2]:
import umap.umap_ as umap
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import random
import statsmodels.api as sm

In [3]:
data = pd.read_csv('edited_data.csv')
data[data < 0] = 0
data.head()

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek
0,0,20140101,1,0,102,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
1,1,20140101,0,0,131,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
2,2,20140101,1,1,91,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
3,3,20140101,0,1,162,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3
4,4,20140101,1,2,86,0.12,1.9,13.0,1.9,1.0,26.0,0.0,0.0,3


In [4]:
data["date"] = pd.to_datetime(data["Date"], format="%Y%m%d")
data["Month"] = data["date"].dt.month
data.head()

,Unnamed: 0,Date,Direction,Time,TRAFFIC,PRCP,SNOW,SNWD,SNOW_DAY_SUM,Vehicles,Driver Age,Condition_Code,MorF,DayOfWeek,date,Month
0,0,20140101,1,0,102,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
1,1,20140101,0,0,131,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
2,2,20140101,1,1,91,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
3,3,20140101,0,1,162,0.12,1.9,13.0,1.9,0.0,0.0,0.0,0.0,3,2014-01-01,1
4,4,20140101,1,2,86,0.12,1.9,13.0,1.9,1.0,26.0,0.0,0.0,3,2014-01-01,1


In [5]:
traffic = data['TRAFFIC']
hour = data['Time']
month = data['Month']
day_of_week = data['DayOfWeek']
snow_sum = data['SNOW_DAY_SUM']
crashed_vehicles = data['Vehicles']

### Making GLM
using poisson

In [6]:
X = pd.DataFrame({
    "hour": data["Time"],
    "month": data["Month"],
    "day_of_week": data["DayOfWeek"],
    "snow_sum": data["SNOW_DAY_SUM"],
    "crashed_vehicles": data["Vehicles"],
})

X["veh_snow_interaction"] = X["crashed_vehicles"] * X["snow_sum"]

X = sm.add_constant(X)

In [7]:
glm_model = sm.GLM(traffic, X)
glm_results = glm_model.fit()

print(glm_results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                TRAFFIC   No. Observations:               167296
Model:                            GLM   Df Residuals:                   167289
Model Family:                Gaussian   Df Model:                            6
Link Function:               Identity   Scale:                      7.2047e+05
Method:                          IRLS   Log-Likelihood:            -1.3656e+06
Date:                Mon, 01 Dec 2025   Deviance:                   1.2053e+11
Time:                        11:14:06   Pearson chi2:                 1.21e+11
No. Iterations:                     3   Pseudo R-squ. (CS):             0.1185
Covariance Type:            nonrobust                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  398.6637 

In [8]:
predicted = glm_results.predict(X)

# Actual values
actual = traffic

# Mean Absolute Error (MAE)
mae = np.mean(np.abs(actual - predicted))

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(np.mean((actual - predicted)**2))

# Pearson correlation between predicted and actual
corr = np.corrcoef(actual, predicted)[0,1]

print("MAE:", mae)
print("RMSE:", rmse)
print("Correlation:", corr)

MAE: 695.2974956073222
RMSE: 848.7895806222137
Correlation: 0.3347054049824444


In [ ]:
deviation_ana = 0
deviation_traffic = 0
deviation_pred = 0

avg_traffic = data['TRAFFIC'].mean()

risk_analysis = 0
t_traffic = 0
t_pred = 0
for x in range(len(data)):
    random_val = random.randint(0, len(data) - 1)
    value = data.iloc[random_val]
    traffic_temp = value['TRAFFIC']
    hour_temp = value['Time']
    month_temp = value['Month']
    day_of_week_temp = value['DayOfWeek']
    snow_sum_temp = value['SNOW_DAY_SUM']
    crashed_vehicles_temp = value['Vehicles']
    # Normal Distribution
    pred_traffic = 398.6637 + (34.1362 * hour_temp) + (-6.4222 * month_temp) + (80.1965 * day_of_week_temp) + (-8.1563 * snow_sum_temp) * (173.8682 * crashed_vehicles_temp) + (-0.1725 * snow_sum_temp * crashed_vehicles_temp)
    
    # Poisson Distribution
    # pred_traffic = 6.3045 + (0.0326 * hour_temp) + (-0.0061 * month_temp) + (0.0758 * day_of_week_temp) + (-0.0085 * snow_sum_temp) * (0.1045 * crashed_vehicles_temp) + (0.0041 * snow_sum_temp * crashed_vehicles_temp)
    # pred_traffic = avg_traffic * 
    risk_analysis += (traffic_temp - pred_traffic)
    t_traffic += traffic_temp
    t_pred += pred_traffic
print("risk_analysis: ", risk_analysis / len(data))
deviation_ana += risk_analysis / len(data)
print("t_traffic: ",t_traffic / len(data))
deviation_traffic += t_traffic / len(data)
print("t_pred: ", pred_traffic / len(data))
deviation_pred += pred_traffic / len(data)
# print('average traffic: ', avg_traffic)
# print('average deviation: ', deviation_ana/15)
# print('deviation traffic: ', deviation_traffic/15)
# print('deviation pred: ', deviation_pred/15)

    

risk_analysis:  417.66329244623404
t_traffic:  1064.8663506599082
t_pred:  0.007923910314651875


In [10]:
y = data["TRAFFIC"]

X = pd.DataFrame({
    "hour": data["Time"],
    "month": data["Month"],
    "day_of_week": data["DayOfWeek"],
    "snow_sum": data["SNOW_DAY_SUM"],
    "crashed_vehicles": data["Vehicles"],
})

X["veh_snow_interaction"] = X["crashed_vehicles"] * X["snow_sum"]

X = sm.add_constant(X)

ols_model = sm.OLS(y, X)
ols_results = ols_model.fit()

print(ols_results.summary())

                            OLS Regression Results                            
Dep. Variable:                TRAFFIC   R-squared:                       0.112
Model:                            OLS   Adj. R-squared:                  0.112
Method:                 Least Squares   F-statistic:                     3518.
Date:                Mon, 01 Dec 2025   Prob (F-statistic):               0.00
Time:                        11:14:27   Log-Likelihood:            -1.3656e+06
No. Observations:              167296   AIC:                         2.731e+06
Df Residuals:                  167289   BIC:                         2.731e+06
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                  398.6637 

In [11]:
import statsmodels.formula.api as smf

# Ensure categorical dtype (optional but good practice)
data["Time"] = data["Time"].astype("category")
data["Month"] = data["Month"].astype("category")
data["DayOfWeek"] = data["DayOfWeek"].astype("category")

# Poisson GLM (counts)
glm_poisson = smf.glm(
    formula="TRAFFIC ~ C(Time) + C(Month) + C(DayOfWeek) + SNOW_DAY_SUM + Vehicles + Vehicles:SNOW_DAY_SUM",
    data=data,
    family=sm.families.Poisson()
).fit()

print(glm_poisson.summary())
pred_glm = glm_poisson.predict(data)

# OLS (general linear model; continuous response)
ols = smf.ols(
    formula="TRAFFIC ~ C(Time) + C(Month) + C(DayOfWeek) + SNOW_DAY_SUM + Vehicles + Vehicles:SNOW_DAY_SUM",
    data=data
).fit()

print(ols.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                TRAFFIC   No. Observations:               167296
Model:                            GLM   Df Residuals:                   167252
Model Family:                 Poisson   Df Model:                           43
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1947e+07
Date:                Mon, 01 Dec 2025   Deviance:                   2.2505e+07
Time:                        11:14:31   Pearson chi2:                 2.44e+07
No. Iterations:                     7   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 5.05

In [12]:
def predict_value(time, month, day_of_week, snow_day_sum, vehicles):
    """
    Predicts the regression outcome based on given inputs.
    
    Parameters:
    - time (int): Time category (0–23, where 0 is baseline and 1–23 are dummy-coded)
    - month (int): Month category (1–12, where 1 is baseline and 2–12 are dummy-coded)
    - day_of_week (int): Day category (1–7, where 1 is baseline and 2–7 are dummy-coded)
    - snow_day_sum (float): Numeric value for snow day sum
    - vehicles (float): Numeric value for vehicles
    
    Returns:
    - float: Predicted regression value
    """
    
    # Intercept
    prediction = 109.2680
    
    # Time effects
    time_coefs = {
        1: -47.9647, 2: -68.6140, 3: -56.4258, 4: 5.2335, 5: 198.8366,
        6: 589.3426, 7: 983.7283, 8: 1213.1129, 9: 1431.3793, 10: 1612.0106,
        11: 1702.8899, 12: 1703.7850, 13: 1774.9735, 14: 1853.5810, 15: 1866.1351,
        16: 1799.4635, 17: 1567.0547, 18: 1233.1918, 19: 888.5421, 20: 621.2204,
        21: 392.6048, 22: 209.9950, 23: 82.4931
    }
    if time in time_coefs:
        prediction += time_coefs[time]
    
    # Month effects
    month_coefs = {
        2: 0.5292, 3: 44.8414, 4: -196.3255, 5: -223.9655, 6: 25.2823,
        7: 191.5921, 8: 89.6680, 9: 46.9572, 10: -171.0176, 11: -227.1095,
        12: -30.4357
    }
    if month in month_coefs:
        prediction += month_coefs[month]
    
    # Day of week effects
    dow_coefs = {
        2: -125.8302, 3: -92.9161, 4: 3.3036, 5: 281.8427,
        6: 296.0086, 7: 339.7153
    }
    if day_of_week in dow_coefs:
        prediction += dow_coefs[day_of_week]
    
    # Continuous predictors
    prediction += snow_day_sum * -2.9470
    prediction += vehicles * 16.5442
    prediction += vehicles * snow_day_sum * -1.3933  # interaction term
    
    return prediction

In [16]:
avg_traffic = data['TRAFFIC'].mean()

risk_analysis = 0
t_traffic = 0
t_pred = 0
for x in range(len(data)):
    random_val = random.randint(0, len(data) - 1)
    value = data.iloc[random_val]
    traffic_temp = value['TRAFFIC']
    hour_temp = value['Time']
    month_temp = value['Month']
    day_of_week_temp = value['DayOfWeek']
    snow_sum_temp = value['SNOW_DAY_SUM']
    crashed_vehicles_temp = value['Vehicles']
    # Normal Distribution
    pred_traffic = predict_value(hour_temp, month_temp, day_of_week_temp, snow_sum_temp, crashed_vehicles_temp)
    
    # Poisson Distribution
    # pred_traffic = 6.3045 + (0.0326 * hour_temp) + (-0.0061 * month_temp) + (0.0758 * day_of_week_temp) + (-0.0085 * snow_sum_temp) * (0.1045 * crashed_vehicles_temp) + (0.0041 * snow_sum_temp * crashed_vehicles_temp)
    # pred_traffic = avg_traffic * 
    risk_analysis += (traffic_temp - pred_traffic) ** 2
    t_traffic += traffic_temp
    t_pred += pred_traffic
print("risk_analysis: ", risk_analysis / len(data))
print("t_traffic: ",t_traffic / len(data))
print("t_pred: ", t_pred / len(data))

risk_analysis:  233660.9979220637
t_traffic:  1066.37484458684
t_pred:  1066.933659753033


In [23]:
random_val = random.randint(0, len(data) - 1)
value = data.iloc[random_val]
traffic_temp = value['TRAFFIC']
hour_temp = value['Time']
month_temp = value['Month']
day_of_week_temp = value['DayOfWeek']
snow_sum_temp = value['SNOW_DAY_SUM']
crashed_vehicles_temp = value['Vehicles']
# Normal Distribution
pred_traffic = predict_value(hour_temp, month_temp, day_of_week_temp, snow_sum_temp, crashed_vehicles_temp)

# Poisson Distribution
# pred_traffic = 6.3045 + (0.0326 * hour_temp) + (-0.0061 * month_temp) + (0.0758 * day_of_week_temp) + (-0.0085 * snow_sum_temp) * (0.1045 * crashed_vehicles_temp) + (0.0041 * snow_sum_temp * crashed_vehicles_temp)
# pred_traffic = avg_traffic * 
print(value)
print((traffic_temp - pred_traffic))
print(traffic_temp)
print(pred_traffic)

Unnamed: 0                       5795
Date                         20140622
Direction                           0
Time                                7
TRAFFIC                           974
PRCP                              0.0
SNOW                              0.0
SNWD                              0.0
SNOW_DAY_SUM                      0.0
Vehicles                          0.0
Driver Age                        0.0
Condition_Code                    0.0
MorF                              0.0
DayOfWeek                           7
date              2014-06-22 00:00:00
Month                               6
Name: 5795, dtype: object
-483.99390000000017
974
1457.9939000000002
